# Hooks 🪝

In [1]:
from tambora.simulation import Sim

def make_sim():
    """A throwaway 1-particle sim reused by the examples below. The cadence
    demos only count *when* hooks fire, so the physics here is irrelevant."""
    sim = Sim()
    sim.add_particles('pts', [[0., 0., 0.]], [[1., 0., 0.]], [1.])
    return sim

## Dictating When a Hook Fires ({class}`~tambora.dynamics.hooks.cadence.Cadence`)

Hooks can vary greatly in the computational expense of their operations and the memory requirements of their stored quantities. For this reason, tambora allows you to determine how frequently each hook fires, giving you the flexibility to record derived quantities at greater a time resolution than the positions, velocities, and masses.

Every hook has a default cadence, which can be changed with the  `cadence=` argument when adding your hook to a {class}`~tambora.simulation.Sim` with {method}`~tambora.simulation.Sim.add_hook()`. If a hook does not have a default cadence, then the hook will fire at every output by default. 

Cadences in tambora operate in terms of either internal integration steps (`steps`, each separated by in time by the `dt` given to{method}`~tambora.simulation.Sim.run`) or outputs (each `dt_out` apart), not physical time. The finest possible resolution is a single step (`dt`). 

Tambora provides the following built-in Cadences:

| Cadence | Fires when | Reach for it when… |
|---|---|---|
| {class}`~tambora.dynamics.hooks.cadence.EveryStep` | every integration step | you need the finest sampling and can afford it |
| {class}`~tambora.dynamics.hooks.cadence.EveryNSteps` | steps where `step % n == 0` | you want a fixed step stride, independent of outputs |
| {class}`~tambora.dynamics.hooks.cadence.EveryOutput` *(default)* | every snapshot step (`step % steps_per_output == 0`) | you want a hook aligned with saved snapshots |
| {class}`~tambora.dynamics.hooks.cadence.EveryNOutputs` | every `n`-th snapshot | you want coarse, cheap sampling |

The example below demonstrates how a hook can be added to a (in this case, arbitrary) simulation with different cadences.

In [7]:
from tambora.dynamics.hooks import EnergyMonitor, EveryStep, EveryOutput

sim = make_sim()

fine, coarse = EnergyMonitor(), EnergyMonitor()
sim.add_hook(fine,   EveryStep())
sim.add_hook(coarse, EveryOutput())

sim.run(t_end=1., dt=0.05, dt_out=0.5, method=None, progress=False)

print(f"EveryStep   fired {len(fine.t)} times")    # 21  (step 0 + 20 steps)
print(f"EveryOutput fired {len(coarse.t)} times")  #  3  (steps 0, 10, 20)

EveryStep   fired 21 times
EveryOutput fired 3 times


The output shows that this choice changes the number of times the hook fires, as expected.

> All built-in cadences in tambora fire at the initial step (`step == 0`) so that hooks capture the initial state of the simulation.

### Making a Custom Cadence

Cadences subclass {class}`~tambora.dynamics.hooks.cadence.Cadence` and implement `due(step, steps_per_output) -> bool` that returns true when the hook should fire. Since cadences are purely step-based, `due` receives
only the current step index and the number of steps between outputs (`dt_out / dt`);
there is no time argument.

As an example, here is a cadence that fires **densely** (every `dt` step) before the first output, then **coarsely** thereafter.

In [3]:
from tambora.dynamics.hooks import Cadence

class DenseThenSparse(Cadence):
    def due(self, step, steps_per_output):
        if step < steps_per_output:          # dense: every step early on
            return True
        return step % steps_per_output == 0  # coarse: only on outputs later

Just like with any other cadences, we can apply the custom cadence to a hook by with `cadence=` in {method}`~tambora.simulation.Sim` with {method}`~tambora.simulation.Sim.add_hook()`

In [4]:
sim = make_sim()

monitor = EnergyMonitor()
sim.add_hook(monitor, DenseThenSparse())

sim.run(t_end=3.0, dt=0.1, dt_out=0.5, method=None, progress=False)
print(f"\nDenseThenSparse fired {len(monitor.t)} times at t = \n")
import numpy as np
for t in monitor.t:
    if np.isclose(t, 0.0):
        print('( ~~~ DENSE firing ~~~ )')
    if np.isclose(t, 1.0):
        print('\n( ~~~ SPARSE firing ~~~ )')
    print(f'\n{t:.1f} Gyr')


DenseThenSparse fired 11 times at t = 

( ~~~ DENSE firing ~~~ )

0.0 Gyr

0.1 Gyr

0.2 Gyr

0.3 Gyr

0.4 Gyr

0.5 Gyr

( ~~~ SPARSE firing ~~~ )

1.0 Gyr

1.5 Gyr

2.0 Gyr

2.5 Gyr

3.0 Gyr


The hook fires on every step through the first output block before
dropping to firing only on output steps, exactly as the schedule `due` encodes.

> *Tip*: Keep `due` cheap since it will run frequently during the simulation.

## Making a Custom Hook

Hooks act on a {class}`~tambora.dynamics.integration.StepState` -- a live, read-only view of the current step. It mirrors the API of the Sim class

In [5]:
from tambora.dynamics.hooks import Hooks

class MyCustomHook(Hook):
    cadence = None

ImportError: cannot import name 'Hooks' from 'tambora.dynamics.hooks' (/geir_data/scr/gabrielspace/tambora/tambora/dynamics/hooks/__init__.py)

## API

```{eval-rst}

.. automethod:: tambora.simulation.Sim.add_hook

.. autoclass:: tambora.dynamics.integration.StepState

.. autoclass:: tambora.dynamics.hooks.cadence.Cadence

.. automethod:: tambora.dynamics.hooks.cadence.Cadence.due

.. autoclass:: tambora.dynamics.hooks.cadence.EveryStep

.. autoclass:: tambora.dynamics.hooks.cadence.EveryNSteps

.. autoclass:: tambora.dynamics.hooks.cadence.EveryOutput

.. autoclass:: tambora.dynamics.hooks.cadence.EveryNOutputs


```